NOTE: Aaj do kaam hain.

- Behavioral features: RFM sirf “kitna, kitni baar, kab” batata hai. Naye features customer ki aadat pakadte hain, jaise “yeh customer aam taur par har 30 din mein aata hai, aur ab 75 din ho gaye”.
- CLV: Customer ki value. Iska matlab yahan “agle 90 din mein expected spend” hai (poori zindagi ka nahi, kyunki 2 saal ke data se lifetime predict karna bharosemand nahi hota).

Leakage ke niyam (wahi, jo ab tak chal rahe hain):

1. Saare naye features sirf cutoff se pehle ke transactions se bante hain.
2. FutureSpend (Day 75 ka target) ko kabhi feature mat banao. Wo sirf evaluation ke liye hai.
3. Imputation aur scaling pipeline ke andar hogi, taaki sirf train se seekhe.
4. Test set sirf aakhir mein ek baar khulega.


1. Behavioral features ka code src/features/behavioral.py mein likho (notebook isko import karega, code do jagah nahi rahega):

In [1]:
"""Behavioural features built ONLY from transactions before the cutoff."""
import numpy as np
import pandas as pd

WINDOW_DAYS = 90


def build_behavioral_features(tx: pd.DataFrame, horizon_days: int = 90) -> pd.DataFrame:
    tx = tx.copy()
    tx['LineTotal'] = tx['Quantity'] * tx['Price']
    cutoff = tx['InvoiceDate'].max() - pd.Timedelta(days=horizon_days)
    past = tx[tx['InvoiceDate'] < cutoff]

    orders = (past.groupby(['Customer ID', 'Invoice'])
              .agg(OrderDate=('InvoiceDate', 'min'), OrderValue=('LineTotal', 'sum'),
                   Items=('Quantity', 'sum'), Lines=('StockCode', 'nunique'))
              .reset_index().sort_values(['Customer ID', 'OrderDate']))
    orders['GapDays'] = orders.groupby('Customer ID')['OrderDate'].diff().dt.total_seconds() / 86400

    g = orders.groupby('Customer ID')
    out = pd.DataFrame({
        'AvgGapDays': g['GapDays'].mean(),
        'StdGapDays': g['GapDays'].std(),
        'AvgItemsPerOrder': g['Items'].mean(),
        'AvgLinesPerOrder': g['Lines'].mean(),
        'MaxOrderValue': g['OrderValue'].max(),
    })

    w = pd.Timedelta(days=WINDOW_DAYS)
    recent = orders[orders['OrderDate'] >= cutoff - w]
    prev = orders[(orders['OrderDate'] >= cutoff - 2 * w) & (orders['OrderDate'] < cutoff - w)]
    out['Orders_recent'] = recent.groupby('Customer ID').size()
    out['Spend_recent'] = recent.groupby('Customer ID')['OrderValue'].sum()
    out['Orders_prev'] = prev.groupby('Customer ID').size()
    cols = ['Orders_recent', 'Spend_recent', 'Orders_prev']
    out[cols] = out[cols].fillna(0)

    out['OrderTrend'] = out['Orders_recent'] - out['Orders_prev']
    out['Spend_recent_log'] = np.log1p(out['Spend_recent'].clip(lower=0))
    out['MaxOrderValue_log'] = np.log1p(out['MaxOrderValue'].clip(lower=0))
    return out.reset_index()

Har feature ka matlab:

- AvgGapDays / StdGapDays: do orders ke beech ka average gap aur uska utar-chadhav (regular ya irregular buyer).
- OrderTrend: pichle 90 din ke orders minus usse pehle ke 90 din ke orders. Negative matlab customer slow ho raha hai.
- Orders_recent / Spend_recent: cutoff se pehle ke aakhri 90 din ki activity.
- AvgItemsPerOrder / AvgLinesPerOrder / MaxOrderValue: basket ka size.

Jo customers ne sirf ek order kiya, unka AvgGapDays NaN hoga (gap banane ke liye do orders chahiye). Ye bug nahi hai, ise aage imputer sambhalega.

2. Is function ke liye test likho (tests/test_behavioral.py):

In [3]:
import pandas as pd
import sys
import os
sys.path.append(os.path.abspath(os.path.join('..', '..')))
from src.features.behavioral import build_behavioral_features


def make_tx():
    rows = [
        (1, 'A1', '2011-01-01', 1, 10.0, 'S1'),
        (1, 'A2', '2011-01-11', 1, 10.0, 'S2'),   # gap = 10 din
        (2, 'A3', '2011-11-25', 1, 50.0, 'S1'),   # sirf future (cutoff = 2011-09-10) -> nahi aana chahiye
        (1, 'A4', '2011-12-09', 5, 100.0, 'S3'),  # future order -> features mein count nahi hona chahiye
    ]
    df = pd.DataFrame(rows, columns=['Customer ID', 'Invoice', 'InvoiceDate', 'Quantity', 'Price', 'StockCode'])
    df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])
    return df


def test_future_only_customers_excluded():
    out = build_behavioral_features(make_tx())
    assert set(out['Customer ID']) == {1}


def test_future_orders_do_not_leak_into_features():
    row = build_behavioral_features(make_tx()).set_index('Customer ID').loc[1]
    assert row['AvgGapDays'] == 10.0      # sirf A1 aur A2 se, future wale A4 se nahi
    assert row['Orders_recent'] == 0

ModuleNotFoundError: No module named 'src.features.behavioral'